In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

missing values in tabular data

In [ ]:
import pandas as pd
from io import StringIO

csv_data = '''A,B,C,D
1.0,2.0,3.0,4.0
5.0,6.0,,8.0
10.0,11.0,12.0,'''

df = pd.read_csv(StringIO(csv_data))
df


In [ ]:
df.isnull().sum()

Eliminating training examples/features w\ missing values

In [ ]:
df.dropna(axis=0) # drop rows

In [ ]:
df.dropna(axis=1) # drop cols

In [ ]:
df.dropna(how='all') # only drops rows that are all NaN

In [ ]:
df.dropna(thresh=4) # drops rows w\ fewer than 4 real values

In [ ]:
df.dropna(subset=['C']) # drops rows where NaN appears in column C

Inputting missing values

In [ ]:
from sklearn.impute import SimpleImputer
import numpy as np
imr = SimpleImputer(missing_values=np.nan, strategy='mean')
imr = imr.fit(df.values)
imputed_data = imr.transform(df.values)
imputed_data

In [ ]:
df.fillna(df.mean()) # alternative to sklearn in pandas

Categorical data encoding

In [ ]:
df = pd.DataFrame([
    ['green', 'M', 10.1, 'class2'],
    ['red', 'L', 13.5, 'class1'],
    ['blue', 'XL', 15.3, 'class2']])
df.columns = ['colour', 'size', 'price', 'classlabel']
df

mapping ordinal data

In [ ]:
size_mapping = {'M': 1,
                'L': 2,
                'XL': 3}
df['size'] = df['size'].map(size_mapping)
df

inverse map for restoring string representation

In [ ]:
inv_size_mapping = {v: k for k, v in size_mapping.items()}
df['size'].map(inv_size_mapping)

encoding nominal class labels (specific number doesn't matter)

In [ ]:
class_mapping = {label: idx for idx, label in 
                 enumerate(np.unique(df['classlabel']))}
class_mapping

In [ ]:
df['classlabel'] = df['classlabel'].map(class_mapping)
df

In [ ]:
inv_class_mapping = {v: k for k, v in class_mapping.items()}
df['classlabel'] = df['classlabel'].map(inv_class_mapping)
df

sci-kit learn encoder

In [ ]:
from sklearn.preprocessing import LabelEncoder
class_le = LabelEncoder()
y = class_le.fit_transform(df['classlabel'].values)
y

In [ ]:
class_le.inverse_transform(y)

### one-hot encoding on nominal features

First: dictionary like approach

In [ ]:
X = df[['colour', 'size', 'price']].values
colour_le = LabelEncoder()
X[:, 0] = colour_le.fit_transform(X[:, 0 ])
X

one-hot encoding: use dummy feature for each new value in the nominal feature column with binary values

In [ ]:
from sklearn.preprocessing import OneHotEncoder
X = df[['colour', 'size', 'price']].values
colour_ohe = OneHotEncoder()
colour_ohe.fit_transform(X[:, 0].reshape(-1, 1)).toarray()

selectively transform features in a multi-feature array

In [ ]:
from sklearn.compose import ColumnTransformer
X = df[['colour', 'size', 'price']].values
c_transf = ColumnTransformer([
    ('onehot', OneHotEncoder(), [0]),
    ('nothing', 'passthrough', [1,2])
])
c_transf.fit_transform(X).astype(float)

convenient way through pandas

In [ ]:
pd.get_dummies(df[['colour', 'size', 'price']])

remove one feature to keep from introducing multi-collinearity (features highly correlated makes them computationally difficult to invert)

In [ ]:
pd.get_dummies(df[['colour', 'size', 'price']], drop_first=True)

dropping redundant via sci-kit learn

In [ ]:
colour_ohe = OneHotEncoder(categories='auto', drop='first')
c_transf = ColumnTransformer([
    ('onehot', colour_ohe, [0]),
    ('nothing', 'passthrough', [1,2])
])
c_transf.fit_transform(X).astype(float)

encoding ordinal features with binary thresholds

In [ ]:
df = pd.DataFrame([
    ['green', 'M', 10.1, 'class2'],
    ['red', 'L', 13.5, 'class1'],
    ['blue', 'XL', 15.3, 'class2']
])
df.columns = ['color', 'size', 'price', 'classlabel']

df['x > M'] = df['size'].apply(lambda x: 1 if x in {'L', 'XL'} else 0)
df['x > L'] = df['size'].apply(lambda x: 1 if x == 'XL' else 0)
del df['size']
df

### Partitioning into train and test datasets

In [ ]:
df_wine = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data', 
                 header=None)
df_wine.columns = [
    'Class label',
    'Alcohol',
    'Malic acid',
    'Ash',
    'Alcalinity of ash',
    'Magnesium',
    'Total phenols',
    'Flavanoids',
    'Nonflavanoid phenols',
    'Proanthocyanins',
    'Color intensity',
    'Hue',
    'OD280/OD315 of diluted wines',
    'Proline'
]
print('Class labels', np.unique(df_wine['Class label']))

In [ ]:
from sklearn.model_selection import train_test_split
X, y = df_wine.iloc[:, 1:].values, df_wine.iloc[:, 0].values
X_train, X_test, y_train, y_test = \
    train_test_split(X, y, 
                     test_size=0.3,
                     random_state=0,
                     stratify=y)


Min-max scaling: $x_{\text{norm}}^{(i)} = \frac{x^{(i)}-x_{\text{min}}^{(i)}}{x_{\text{max}}^{(i)}-x_{\text{min}}^{(i)}}$

Normalization: min-max scaling on $[0,1]$

In [ ]:
from sklearn.preprocessing import MinMaxScaler
mms = MinMaxScaler()
X_train_norm = mms.fit_transform(X_train)
X_test_norm = mms.fit(X_test)

Standardization: $x_{\text{std}}^{(i)} = \frac{x^{(i)}-\mu_x}{\sigma_x}$

In [ ]:
from sklearn.preprocessing import StandardScaler
stdsc = StandardScaler()
X_train_std = stdsc.fit_transform(X_train)
X_test_std = stdsc.transform(X_test)

RobustScaler can be a good option for small datasets or models prone to overfitting

### L1 and L2 Regularization
$$L2: ||\mathbf{w}||_2^2 = \sum_{j=1}^m w_j^2$$
Used to reduce complexity of a model by penalizing larger weights
$$L1: ||\mathbf{w}||_1 = \sum_{j=1}^m |w_j|$$
Yields sparse feature vector (can be useful in high dimensions with more irrelevant features), used as a technique for feature selection

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
lr = OneVsRestClassifier(
    LogisticRegression(
        C=1.0,
        solver='liblinear',
        l1_ratio=1
    )
)

lr.fit(X_train_std, y_train)
print(f"Training accuracy: {lr.score(X_train_std, y_train)}")
print(f"Testing accuracy: {lr.score(X_test_std, y_test)}")

In [ ]:
[est.intercept_ for est in lr.estimators_]

In [ ]:
[est.coef_ for est in lr.estimators_]

In [ ]:
import matplotlib.pyplot as plt
fig = plt.figure()
ax = plt.subplot(111)
colours = [
    'blue', 'green', 'red', 'cyan',
    'magenta', 'yellow', 'black',
    'pink', 'lightgreen', 'lightblue',
    'gray', 'indigo', 'orange'
]
weights, params = [], []
for c in np.arange(-4., 6.):
    lr = OneVsRestClassifier(
        LogisticRegression(
            C=10.**c,
            solver='liblinear',
            l1_ratio=1,
            random_state=0
        )
    )
    lr.fit(X_train_std, y_train)
    w = np.vstack([est.coef_ for est in lr.estimators_])
    weights.append(w[1])
    params.append(10**c)

weights = np.array(weights)
for col, colour in zip(range(weights.shape[1]), colours):
    plt.plot(params, weights[:, col],
             label=df_wine.columns[col + 1],
             color=colour)

plt.axhline(0, color='black', linestyle='--', linewidth=3)
plt.xlim([10**(-5), 10**5])
plt.ylabel('Weight coefficient')
plt.xlabel('C (inverse regularization strength)')
plt.xscale('log')
plt.legend(loc='upper left')
ax.legend(loc='upper center',
          bbox_to_anchor=(1.38, 1.03),
          ncol=1, fancybox=True)
plt.show()
    

### Sequential feature selection algorithms
**Feature selection**: select a subset of original features\
<br>
**Feature extraction**: derive information from features to construct a new feature subspace\
<br>
these algorithms are a family of greedy search algorithms\
classically, **sequential backward selection (SBS)** which needs criterion function $J$ to minimize
<br>
1. initialize $k=d$ where $d$ is the dimensionality of full feature space $\mathbf{X}_d$
2. $\mathbf x ^- = \text{argmax} J (\mathbf X_k - \mathbf x)$, where $\mathbf x \in \mathbf X_k$
3. $\mathbf X_{k-1}=\mathbf X_k - \mathbf x^-; k = k - 1$
4. repeat steps 2-3 until reached desired # of features

In [ ]:
from utils import SBS
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
sbs = SBS(knn, k_features=1)
sbs.fit(X_train_std, y_train)
k_feat = [len(k) for k in sbs.subsets_]
plt.plot(k_feat, sbs.scores_, marker='o')
plt.ylim([0.7, 1.02])
plt.ylabel('Accuracy')
plt.xlabel('Number of features')
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
k5 = list(sbs.subsets_[8])
print(df_wine.columns[1:][k5])

In [ ]:
k3 = list(sbs.subsets_[10])
print(df_wine.columns[1:][k3])

In [ ]:
knn.fit(X_train_std, y_train)
print(knn.score(X_train_std, y_train))
print(knn.score(X_test_std, y_test))

In [ ]:
knn.fit(X_train_std[:, k5], y_train)
print('Training accuracy:', knn.score(X_train_std[:, k5], y_train))
print('Test accuracy:', knn.score(X_test_std[:, k5], y_test))

In [ ]:
knn.fit(X_train_std[:, k3], y_train)
print('Training accuracy:', knn.score(X_train_std[:, k3], y_train))
print('Test accuracy:', knn.score(X_test_std[:, k3], y_test))

In [ ]:
print("k3 indices:", k3)
print("k5 indices:", k5)

knn.fit(X_train_std[:, k3], y_train)
pred_k3 = knn.predict(X_test_std[:, k3])

knn.fit(X_train_std[:, k5], y_train)
pred_k5 = knn.predict(X_test_std[:, k5])

print("Different predictions:", np.sum(pred_k3 != pred_k5))
print("Correct predictions for k3:", np.sum(pred_k3 == y_test))
print("Correct predictions for k5:", np.sum(pred_k5 == y_test))